In [1]:
# [Cell 1 - Code]
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import pickle

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
import numpy as np
import pandas as pd

# 1. Set seed for reproducibility
np.random.seed(42)
num_rows = 10000

print(f"Generating {num_rows} rows of game telemetry data...")

# 2. Generate random features representing city states
# Simulated range: 0 to 600 points of health/environmental damage
damage_taken = np.random.randint(0, 601, num_rows)

# Simulated range: 0 to 45 active enemies attacking the city
enemies_spawned = np.random.randint(0, 46, num_rows)

# Simulated range: 0 to 8 structural assets destroyed
structures_destroyed = np.random.randint(0, 9, num_rows)

# 3. Define the ground-truth relationship for Danger %
# We apply logical weights: structures hit hardest, then enemy count, then raw damage
base_danger = (
    (structures_destroyed * 6.5) + (enemies_spawned * 1.2) + (damage_taken * 0.08)
)

# Add Gaussian noise to prevent the model from learning a trivial, perfect equation.
# This simulates unpredictable game conditions (e.g., player level, map layout).
noise = np.random.normal(loc=0.0, scale=4.0, size=num_rows)
final_danger = base_danger + noise

# Bound the target variable strictly between 0% and 100%
final_danger = np.clip(final_danger, 0.0, 100.0)

# 4. Construct the DataFrame
df = pd.DataFrame(
    {
        "damage": damage_taken,
        "enemies": enemies_spawned,
        "structures_destroyed": structures_destroyed,
        "danger_percent": np.round(final_danger, 2),
    }
)

# 5. Export to CSV file
csv_filename = "city_danger_data.csv"
df.to_csv(csv_filename, index=False)

print(f"Success! Saved dataset to '{csv_filename}'")
print("\nData preview:")
print(df.head())

Generating 10000 rows of game telemetry data...
Success! Saved dataset to 'city_danger_data.csv'

Data preview:
   damage  enemies  structures_destroyed  danger_percent
0     102        0                     1           20.32
1     435        2                     7           86.70
2     270       25                     0           52.79
3     106       30                     7           89.13
4      71       25                     2           52.09


In [4]:
# [Cell 2 - Code]
# Load the CSV file
# Replace 'city_danger_data.csv' with the actual name of your file
df = pd.read_csv('city_danger_data.csv')

# Verify the data loaded correctly
print(f"Total rows loaded: {len(df)}")
df.head() # Displays the first 5 rows to ensure columns match

Total rows loaded: 10000


,damage,enemies,structures_destroyed,danger_percent
0,102,0,1,20.32
1,435,2,7,86.70
2,270,25,0,52.79
3,106,30,7,89.13
4,71,25,2,52.09


In [8]:
# [Cell 3 - Code]
# 1. Separate your inputs (Features) from your output (Target)
X = df[['damage', 'enemies', 'structures_destroyed']]
y = df['danger_percent']

# 2. Split the data (8,000 rows for training, 2,000 rows for testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and Train the Random Forest



In [9]:
# [Cell - Multi-Model Training & Evaluation]
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import pickle

# 1. Define the models we want to compare in a dictionary
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=5)
}

best_model = None
best_model_name = ""
best_mae = float('inf') # Start with an infinitely high error

print("⚔️ Model Arena: Training & Evaluating...\n" + "="*40)

# 2. Loop through the dictionary, train, and test each one
for name, model in models.items():
    # Train the model on the 80% data
    model.fit(X_train, y_train)
    
    # Predict on the 20% unseen data
    predictions = model.predict(X_test)
    
    # Calculate performance metrics
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    
    print(f"--- {name} ---")
    print(f"Mean Absolute Error: {mae:.2f}%")
    print(f"R² Score: {r2:.4f}\n")
    
    # 3. Automatically capture the champion model (Lowest Error)
    if mae < best_mae:
        best_mae = mae
        best_model_name = name
        best_model = model

# 4. Declare the winner and export it
print("="*40)
print(f"🏆 WINNER: {best_model_name} (MAE: {best_mae:.2f}%)")

# Save the absolute best model for Godot to use
export_filename = 'best_danger_model.pkl'
with open(export_filename, 'wb') as file:
    pickle.dump(best_model, file)

print(f"💾 '{export_filename}' successfully exported! Ready for the Flask API.")

⚔️ Model Arena: Training & Evaluating...
--- Linear Regression ---
Mean Absolute Error: 5.16%
R² Score: 0.9107

--- Random Forest ---
Mean Absolute Error: 3.02%
R² Score: 0.9657

--- Gradient Boosting ---
Mean Absolute Error: 3.19%
R² Score: 0.9681

--- K-Nearest Neighbors ---
Mean Absolute Error: 5.17%
R² Score: 0.9073

🏆 WINNER: Random Forest (MAE: 3.02%)
💾 'best_danger_model.pkl' successfully exported! Ready for the Flask API.
